<a href="https://colab.research.google.com/github/Marfall/CNN-Otus-6/blob/main/CNN_Otus_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Домашнее задание №6: Классификация организации облаков

**Цель:** Построить свёрточную нейронную сеть (U-Net) для сегментации облачных структур на спутниковых снимках. Оценить качество с помощью Dice coefficient.

**Датасет:** Understanding Cloud Organization (4 класса: Fish, Flower, Gravel, Sugar).

**План:**
1. Импорт библиотек, настройка окружения.
2. Загрузка и анализ структуры данных.
3. EDA: распределение классов, визуализация.
4. Формирование подвыборки для ускоренного обучения.
5. Dataset/DataLoader с ресайзом и нормализацией.
6. Архитектура U-Net (4-канальный выход).
7. Функция потерь: BCE + Dice. Метрика: Dice coefficient.
8. Обучение с ранней остановкой по val_dice.
9. Оценка, визуализация предсказаний.
10. Выводы.

## 1. Импорт библиотек и настройка окружения

Подключаем стандартный стек: PyTorch, NumPy, pandas для табличных сводок, Seaborn/Matplotlib для визуализации. Настраиваем единый стиль графиков.

In [ ]:
# 2: Импорт библиотек и настройка стиля визуализации
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Единый стиль графиков
sns.set_style('whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 3. Фиксация seed и определение устройства

Фиксируем seed для numpy и torch — это обеспечивает воспроизводимость результатов. Определяем устройство: GPU, если доступен, иначе CPU.

In [ ]:
# 4: Фиксация seed и выбор устройства
def set_seed(seed=42):
    """Фиксирует seed для всех генераторов случайных чисел."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 5. Загрузка данных

Данные распакованы в `/content/cloud_dhw/cloud_data`. Проверяем структуру папок и находим пути к изображениям и маскам.

In [ ]:
# 6: Проверка структуры распакованных данных
import shutil

extract_dir = '/content/cloud_dhw/cloud_data'

# Вывод структуры верхних уровней
print('=== Структура данных ===')
for root, dirs, files in os.walk(extract_dir):
    level = root.replace(extract_dir, '').count(os.sep)
    if level < 3:
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/  [{len(files)} файлов]')

# Свободное место
total, used, free = shutil.disk_usage('/content')
print(f'\nСвободно в /content: {free / 1e9:.1f} ГБ')

## 7. Формирование подвыборки

Полный датасет содержит несколько тысяч изображений высокого разрешения. Для ускоренного обучения и укладывания в лимиты Colab формируем подвыборку: случайные N изображений из тренировочной выборки. Подвыборку сохраняем на Google Drive — для повторного использования без перекачивания.

In [ ]:
# 8: Формирование подвыборки и сохранение на Google Drive
from google.colab import drive

# Монтируем Drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print('Google Drive смонтирован.')
else:
    print('Google Drive уже смонтирован.')

# Ищем папки с изображениями и масками (адаптивно)
# Предполагаем наличие train_images и train_masks
candidates_img = []
candidates_msk = []
for root, dirs, files in os.walk(extract_dir):
    name = os.path.basename(root).lower()
    if 'image' in name and any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in files):
        candidates_img.append(root)
    if 'mask' in name and any(f.lower().endswith(('.png', '.jpg')) for f in files):
        candidates_msk.append(root)

print(f'Найдено папок с изображениями: {candidates_img}')
print(f'Найдено папок с масками: {candidates_msk}')

## 9. Разведочный анализ данных (EDA)

Анализируем: количество изображений, размеры, распределение по классам. Визуализируем случайные примеры «изображение + маски».

In [ ]:
# 10: Базовая статистика по датасету
# Работаем с первой найденной папкой изображений
IMG_DIR = candidates_img[0] if candidates_img else None
MSK_DIR = candidates_msk[0] if candidates_msk else None

if IMG_DIR:
    img_files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'Папка изображений: {IMG_DIR}')
    print(f'Всего изображений: {len(img_files)}')

    # Размеры нескольких случайных изображений
    sample_sizes = []
    for f in img_files[:10]:
        with Image.open(os.path.join(IMG_DIR, f)) as im:
            sample_sizes.append(im.size)
    sizes_df = pd.DataFrame({'Размер (WxH)': sample_sizes})
    print('\n=== Размеры случайных 10 изображений ===')
    print(sizes_df.to_string(index=False))
else:
    print('Папка изображений не найдена — проверь структуру.')

if MSK_DIR:
    msk_files = sorted(os.listdir(MSK_DIR))
    print(f'\nПапка масок: {MSK_DIR}')
    print(f'Всего файлов масок: {len(msk_files)}')

In [ ]:
# 11: Распределение классов (по суффиксам масок)
CLASSES = ['Fish', 'Flower', 'Gravel', 'Sugar']

if MSK_DIR:
    # Считаем, сколько масок каждого класса
    class_counts = {c: 0 for c in CLASSES}
    for f in msk_files:
        for c in CLASSES:
            if c.lower() in f.lower():
                class_counts[c] += 1
                break

    counts_df = pd.DataFrame({
        'Класс': list(class_counts.keys()),
        'Кол-во масок': list(class_counts.values())
    })
    print('=== Распределение масок по классам ===')
    print(counts_df.to_string(index=False))

    # Визуализация
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=counts_df, x='Класс', y='Кол-во масок', ax=ax)
    ax.set_title('Распределение масок по классам')
    ax.set_ylabel('Количество')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    plt.close('all')

In [ ]:
# 12: Визуализация примеров (изображение + 4 маски)
if IMG_DIR and MSK_DIR:
    sample_imgs = random.sample(img_files, min(3, len(img_files)))

    fig, axes = plt.subplots(len(sample_imgs), 5, figsize=(15, 3 * len(sample_imgs)))

    for i, fname in enumerate(sample_imgs):
        # Изображение
        img_path = os.path.join(IMG_DIR, fname)
        img = Image.open(img_path).convert('RGB')
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'Оригинал\n{fname[:20]}')
        axes[i, 0].axis('off')

        # 4 маски
        base = os.path.splitext(fname)[0]
        for j, cls in enumerate(CLASSES):
            # Пробуем найти маску по суффиксу класса
            mask_name = None
            for mf in msk_files:
                if base in mf and cls.lower() in mf.lower():
                    mask_name = mf
                    break
            if mask_name:
                mask = Image.open(os.path.join(MSK_DIR, mask_name)).convert('L')
                axes[i, j + 1].imshow(mask, cmap='gray')
                axes[i, j + 1].set_title(cls)
            else:
                axes[i, j + 1].text(0.5, 0.5, 'нет', ha='center', va='center')
                axes[i, j + 1].set_title(cls)
            axes[i, j + 1].axis('off')

    plt.tight_layout()
    plt.show()
    plt.close('all')

## 13. Выводы по EDA

**Заполнить после запуска ячеек выше.** Ниже — плейсхолдеры для чисел, которые нужно вставить вручную или через Кэпа.

- Всего изображений: `[N_IMAGES]`
- Размер изображений: `[IMG_SIZE]`
- Распределение классов: `[CLASS_DIST]`
- Дисбаланс классов: `[IMBALANCE]`

## 14. Dataset и DataLoader

Создаём кастомный `Dataset`: загружает изображение, ресайзит до 256×256, нормализует. Маску собирает из 4 PNG-файлов в 4-канальный тензор (по одному каналу на класс). Применяем простые аугментации (случайные флипы).

In [ ]:
# 15: Класс Dataset для сегментации облаков
class CloudDataset(Dataset):
    def __init__(self, img_dir, msk_dir, img_files, classes, img_size=256, augment=False):
        self.img_dir = img_dir
        self.msk_dir = msk_dir
        self.img_files = img_files
        self.classes = classes
        self.img_size = img_size
        self.augment = augment
        # Индекс масок по имени для быстрого поиска
        self.msk_lookup = os.listdir(msk_dir)

    def __len__(self):
        return len(self.img_files)

    def _find_mask(self, base_name, cls):
        """Находит имя файла маски по базовому имени изображения и классу."""
        for mf in self.msk_lookup:
            if base_name in mf and cls.lower() in mf.lower():
                return mf
        return None

    def __getitem__(self, idx):
        fname = self.img_files[idx]
        base = os.path.splitext(fname)[0]

        # Изображение
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img = np.array(img, dtype=np.float32) / 255.0
        img = img.transpose(2, 0, 1)  # HWC -> CHW

        # Маска: 4 канала
        mask = np.zeros((len(self.classes), self.img_size, self.img_size), dtype=np.float32)
        for i, cls in enumerate(self.classes):
            mf = self._find_mask(base, cls)
            if mf is not None:
                m = Image.open(os.path.join(self.msk_dir, mf)).convert('L')
                m = m.resize((self.img_size, self.img_size), Image.NEAREST)
                m = np.array(m, dtype=np.float32) / 255.0
                mask[i] = (m > 0.5).astype(np.float32)

        # Аугментации: случайные флипы
        if self.augment:
            if random.random() < 0.5:
                img = img[:, :, ::-1].copy()
                mask = mask[:, :, ::-1].copy()
            if random.random() < 0.5:
                img = img[:, ::-1, :].copy()
                mask = mask[:, ::-1, :].copy()

        return torch.from_numpy(img), torch.from_numpy(mask)

print('Класс CloudDataset определён.')

In [ ]:
# 16: Train/Val split и DataLoader
if IMG_DIR:
    random.seed(42)
    random.shuffle(img_files)

    # Берём подвыборку для скорости (можно увеличить)
    SUBSET_SIZE = min(600, len(img_files))
    subset_files = img_files[:SUBSET_SIZE]

    split = int(0.8 * len(subset_files))
    train_files = subset_files[:split]
    val_files = subset_files[split:]

    print(f'Подвыборка: {SUBSET_SIZE}')
    print(f'Train: {len(train_files)} | Val: {len(val_files)}')

    IMG_SIZE = 256
    BATCH_SIZE = 16

    train_ds = CloudDataset(IMG_DIR, MSK_DIR, train_files, CLASSES, img_size=IMG_SIZE, augment=True)
    val_ds = CloudDataset(IMG_DIR, MSK_DIR, val_files, CLASSES, img_size=IMG_SIZE, augment=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    # Проверка одного батча
    xb, yb = next(iter(train_loader))
    print(f'\nФорма батча изображений: {xb.shape}')
    print(f'Форма батча масок: {yb.shape}')

## 17. Архитектура U-Net

Классическая U-Net: энкодер из 4 блоков (Conv-BN-ReLU + MaxPool), bottleneck, декодер с skip-connections (ConvTranspose + конкатенация с энкодером). Выход — 4 канала (по классу), активация не применяется (используем BCEWithLogitsLoss).

In [ ]:
# 18: Архитектура U-Net
class DoubleConv(nn.Module):
    """Двойная свёртка: Conv -> BN -> ReLU -> Conv -> BN -> ReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    """U-Net с 4 уровнями энкодера/декодера."""
    def __init__(self, in_ch=3, out_ch=4, base=32):
        super().__init__()
        # Энкодер
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)

        # Bottleneck
        self.bottleneck = DoubleConv(base * 8, base * 16)

        # Декодер
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = DoubleConv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)

        # Выход
        self.out = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.out(d1)

model = UNet(in_ch=3, out_ch=len(CLASSES), base=32).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net создан. Обучаемых параметров: {n_params:,}')

## 19. Функция потерь и метрика

**Loss:** комбинация `BCEWithLogitsLoss` (попиксельная) и `Dice Loss` (структурная). Dice Loss = 1 - Dice coefficient. Комбинация устойчива к дисбалансу классов (облака занимают малую часть пикселей).

**Метрика:** Dice coefficient — основная по ТЗ.

In [ ]:
# 20: Dice coefficient, Dice Loss, комбинированная функция потерь
def dice_coef(preds, targets, eps=1e-6):
    """Dice coefficient, усреднённый по всем каналам и батчу."""
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).float()
    intersection = (preds * targets).sum(dim=(2, 3))
    union = preds.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean()

class DiceBCELoss(nn.Module):
    """Комбинация BCE и Dice Loss."""
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)
        preds_sig = torch.sigmoid(preds)
        intersection = (preds_sig * targets).sum(dim=(2, 3))
        union = preds_sig.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice_loss = 1 - ((2 * intersection + 1e-6) / (union + 1e-6)).mean()
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss

criterion = DiceBCELoss(bce_weight=0.5)
print('Функция потерь DiceBCELoss инициализирована.')

## 21. Обучение с ранней остановкой

Оптимизатор Adam, lr=1e-3, Mixed Precision (AMP) для ускорения на T4, ранняя остановка по `val_dice` с patience=7. Лучшая модель сохраняется в памяти и восстанавливается после остановки.

In [ ]:
# 22: Цикл обучения с ранней остановкой
from torch.cuda.amp import autocast, GradScaler

EPOCHS = 40
PATIENCE = 7
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler = GradScaler() if device.type == 'cuda' else None

history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}
best_dice = 0.0
best_state = None
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss_sum, train_dice_sum = 0.0, 0.0
    for xb, yb in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [train]', leave=False):
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        if scaler is not None:
            with autocast():
                logits = model(xb)
                loss = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
        train_loss_sum += loss.item() * xb.size(0)
        train_dice_sum += dice_coef(logits.detach(), yb).item() * xb.size(0)

    train_loss = train_loss_sum / len(train_ds)
    train_dice = train_dice_sum / len(train_ds)

    # --- Val ---
    model.eval()
    val_loss_sum, val_dice_sum = 0.0, 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss_sum += loss.item() * xb.size(0)
            val_dice_sum += dice_coef(logits, yb).item() * xb.size(0)

    val_loss = val_loss_sum / len(val_ds)
    val_dice = val_dice_sum / len(val_ds)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)

    print(f'Epoch {epoch:02d} | train_loss={train_loss:.4f} train_dice={train_dice:.4f} | val_loss={val_loss:.4f} val_dice={val_dice:.4f}')

    # Ранняя остановка
    if val_dice > best_dice:
        best_dice = val_dice
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nРанняя остановка на эпохе {epoch}. Лучший val_dice = {best_dice:.4f}')
            break

# Восстанавливаем лучшую модель
if best_state is not None:
    model.load_state_dict(best_state)
    print(f'\nВосстановлена лучшая модель. Best val_dice = {best_dice:.4f}')

## 23. Графики обучения

Строим динамику loss и dice по эпохам. Под графиками — числовая сводка для копирования.

In [ ]:
# 24: Графики обучения
epochs_range = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], label='train', marker='o')
axes[0].plot(epochs_range, history['val_loss'], label='val', marker='s')
axes[0].set_title('Loss по эпохам')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history['train_dice'], label='train', marker='o')
axes[1].plot(epochs_range, history['val_dice'], label='val', marker='s')
axes[1].set_title('Dice coefficient по эпохам')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Dice')
axes[1].legend()

plt.tight_layout()
plt.show()
plt.close('all')

# Числовая сводка
hist_df = pd.DataFrame({
    'Эпоха': list(epochs_range),
    'train_loss': [f'{v:.4f}' for v in history['train_loss']],
    'val_loss': [f'{v:.4f}' for v in history['val_loss']],
    'train_dice': [f'{v:.4f}' for v in history['train_dice']],
    'val_dice': [f'{v:.4f}' for v in history['val_dice']],
})
print('=== История обучения ===')
print(hist_df.to_string(index=False))

## 25. Оценка на валидационной выборке

Финальная оценка модели: Dice coefficient по каждому классу отдельно, плюс средний. Используем лучшую (восстановленную) модель.

In [ ]:
# 26: Оценка Dice по классам на валидации
model.eval()

# Накапливаем предсказания и таргеты
all_preds = []
all_targets = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = (torch.sigmoid(logits) > 0.5).float().cpu()
        all_preds.append(preds)
        all_targets.append(yb)

all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

# Dice по каждому классу
dice_per_class = {}
for i, cls in enumerate(CLASSES):
    p = all_preds[:, i]
    t = all_targets[:, i]
    inter = (p * t).sum().item()
    union = p.sum().item() + t.sum().item()
    dice_per_class[cls] = (2 * inter + 1e-6) / (union + 1e-6)

mean_dice = np.mean(list(dice_per_class.values()))

dice_df = pd.DataFrame({
    'Класс': list(dice_per_class.keys()),
    'Dice': [f'{v:.4f}' for v in dice_per_class.values()]
})
print('=== Dice по классам ===')
print(dice_df.to_string(index=False))
print(f'\nСредний Dice: {mean_dice:.4f}')

# Визуализация
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(x=list(dice_per_class.keys()), y=list(dice_per_class.values()), ax=ax)
ax.set_title(f'Dice coefficient по классам (средний = {mean_dice:.4f})')
ax.set_ylabel('Dice')
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
plt.close('all')

# Переменные для выводов
DICE_FISH = dice_per_class['Fish']
DICE_FLOWER = dice_per_class['Flower']
DICE_GRAVEL = dice_per_class['Gravel']
DICE_SUGAR = dice_per_class['Sugar']
MEAN_DICE = mean_dice
BEST_VAL_DICE = best_dice
N_EPOCHS_RUN = len(history['train_loss'])

## 27. Визуализация предсказаний

Сравниваем на валидационных примерах: оригинал, ground truth, предсказание модели. Это даёт качественное представление о работе сети.

In [ ]:
# Ячейка 28: Визуализация предсказаний модели
model.eval()

# Берём несколько примеров из валидации
n_show = 3
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))

with torch.no_grad():
    for i in range(n_show):
        xb, yb = val_ds[i]
        xb_dev = xb.unsqueeze(0).to(device)
        logits = model(xb_dev)
        pred = (torch.sigmoid(logits) > 0.5).float().cpu().squeeze(0)

        # Оригинал
        img_show = xb.permute(1, 2, 0).numpy()
        axes[i, 0].imshow(img_show)
        axes[i, 0].set_title('Оригинал')
        axes[i, 0].axis('off')

        # GT: объединение всех классов в RGB-подобную картинку
        gt_show = yb.permute(1, 2, 0).numpy()
        axes[i, 1].imshow(gt_show[:, :, :3] if gt_show.shape[2] >= 3 else gt_show[:, :, 0], cmap=None)
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')

        # Pred
        pred_show = pred.permute(1, 2, 0).numpy()
        axes[i, 2].imshow(pred_show[:, :, :3] if pred_show.shape[2] >= 3 else pred_show[:, :, 0], cmap=None)
        axes[i, 2].set_title('Предсказание')
        axes[i, 2].axis('off')

plt.tight_layout()
plt.show()
plt.close('all')

## 30. Выводы

**Заполнить после получения чисел.** Плейсхолдеры:

- Лучший val_dice: `[BEST_VAL_DICE]` на эпохе `[N_EPOCHS_RUN]`
- Средний Dice на валидации: `[MEAN_DICE]`
- По классам: Fish = `[DICE_FISH]`, Flower = `[DICE_FLOWER]`, Gravel = `[DICE_GRAVEL]`, Sugar = `[DICE_SUGAR]`
- Наблюдения по графику обучения: `[TRAIN_OBS]`
- Слабые классы: `[WEAK_CLASS]`
- Возможные улучшения: `[IMPROVEMENTS]`

In [ ]:
# 30: Финальная сводка для передачи данных
final_summary = pd.DataFrame({
    'Параметр': [
        'Размер подвыборки',
        'Train / Val',
        'Эпох обучено',
        'Лучший val_dice',
        'Средний Dice (val)',
        'Dice Fish',
        'Dice Flower',
        'Dice Gravel',
        'Dice Sugar',
        'Размер изображений',
        'Batch size',
    ],
    'Значение': [
        SUBSET_SIZE,
        f'{len(train_files)} / {len(val_files)}',
        N_EPOCHS_RUN,
        f'{BEST_VAL_DICE:.4f}',
        f'{MEAN_DICE:.4f}',
        f'{DICE_FISH:.4f}',
        f'{DICE_FLOWER:.4f}',
        f'{DICE_GRAVEL:.4f}',
        f'{DICE_SUGAR:.4f}',
        f'{IMG_SIZE}x{IMG_SIZE}',
        BATCH_SIZE,
    ]
})
print('=== Финальная сводка ===')
print(final_summary.to_string(index=False))